# Structural quality and false-negative review
Elham Hadadi — Laboratory of Bioinformatics I

Training quality is recalculated from the52 gap-free MUSTANG columns. The second calculation independently verifies native ChimeraX results and source sequence identity. Install the listed Python requirements and use this folder as the notebook working directory. To repeat the native Matchmaker step, use run_chimerax.py with ChimeraX1.11.1; it is not executed by this notebook.


In [1]:
from pathlib import Path
import csv,json,itertools,hashlib
import numpy as np
from Bio.PDB import PDBParser
from Bio.SeqUtils import seq1
BASE=Path.cwd();I=BASE/'inputs';O=BASE/'results';O.mkdir(exist_ok=True)
seqs={};key=None
for l in (I/'seed_alignment.fasta').read_text().splitlines():
 if l.startswith('>'):key=l[1:].split()[0];seqs[key]=''
 else:seqs[key]+=l.strip()
assert len(seqs)==13 and {len(s) for s in seqs.values()}=={53}
columns=[i for i in range(53) if all(seqs[k][i] not in '-.' for k in seqs)];assert len(columns)==52
parser=PDBParser(QUIET=True);xyz={};length={}
for k,s in seqs.items():
 residues=[r for r in parser.get_structure(k,I/'seed_fragments'/(k+'.pdb')).get_residues() if 'CA' in r]
 pdbseq=''.join(seq1(r.resname) for r in residues);assert pdbseq==s.replace('-','').replace('.',''),k
 col_to_res={};j=0
 for i,a in enumerate(s):
  if a not in '-.':col_to_res[i]=j;j+=1
 xyz[k]=np.array([residues[col_to_res[i]]['CA'].coord.astype(float) for i in columns]);length[k]=len(residues)
old={tuple(sorted([r['seed_a'],r['seed_b']])):float(r['pairwise_fitted_rmsd_angstrom']) for r in csv.DictReader((I/'archived_pairwise_rmsd.csv').open())}
rows=[]
for a,b in itertools.combinations(sorted(seqs),2):
 x,y=xyz[a],xyz[b];xc=x-x.mean(0);yc=y-y.mean(0);u,sv,vt=np.linalg.svd(xc.T@yc);rot=u@vt
 if np.linalg.det(rot)<0:u[:,-1]*=-1;rot=u@vt
 rms=float(np.sqrt(np.mean(np.sum((xc@rot-yc)**2,axis=1))));q=52**2/((1+(rms/3)**2)*length[a]*length[b])
 assert abs(rms-old[(a,b)])<1e-5,(a,b,rms,old[(a,b)])
 rows.append(dict(seed_a=a,seed_b=b,aligned_CA_pairs=52,N1=length[a],N2=length[b],rmsd_angstrom=rms,Q_score=q,archived_rmsd_angstrom=old[(a,b)]))
assert len(rows)==78
with (O/'training_pairwise_quality.csv').open('w') as f:w=csv.DictWriter(f,fieldnames=list(rows[0]));w.writeheader();w.writerows(rows)
meta={r['seed_id']:r for r in csv.DictReader((I/'fragment_manifest.csv').open())};summary=[]
for a in sorted(seqs):
 r=[r for r in rows if a in [r['seed_a'],r['seed_b']]];m=meta[a]
 summary.append(dict(seed_id=a,pdb=m['entity_identifier'].split('_')[0],chain=m['source_label_chain'],method=m['experimental_method'],length=length[a],mean_pairwise_rmsd_angstrom=float(np.mean([x['rmsd_angstrom'] for x in r])),maximum_pairwise_rmsd_angstrom=max(x['rmsd_angstrom'] for x in r),mean_pairwise_Q=float(np.mean([x['Q_score'] for x in r]))))
with (O/'training_quality_summary.csv').open('w') as f:w=csv.DictWriter(f,fieldnames=list(summary[0]));w.writeheader();w.writerows(summary)
result=dict(training_fragments=13,alignment_columns=53,common_gap_free_columns=52,pairs_recomputed=78,maximum_archive_difference=max(abs(r['rmsd_angstrom']-r['archived_rmsd_angstrom']) for r in rows),mean_pairwise_rmsd_angstrom=float(np.mean([r['rmsd_angstrom'] for r in rows])),minimum_pairwise_rmsd_angstrom=min(r['rmsd_angstrom'] for r in rows),maximum_pairwise_rmsd_angstrom=max(r['rmsd_angstrom'] for r in rows),mean_pairwise_Q=float(np.mean([r['Q_score'] for r in rows])),minimum_pairwise_Q=min(r['Q_score'] for r in rows),maximum_pairwise_Q=max(r['Q_score'] for r in rows),Q_definition='Nalign^2 / ((1+(RMSD/3A)^2)*N1*N2), using fixed MUSTANG correspondence; not a new PDBeFold alignment',source_coordinates_modified=False,training_membership_modified=False)
(O/'training_quality_results.json').write_text(json.dumps(result,indent=2));print(json.dumps(result,indent=2))


{
  "training_fragments": 13,
  "alignment_columns": 53,
  "common_gap_free_columns": 52,
  "pairs_recomputed": 78,
  "maximum_archive_difference": 2.55351295663786e-15,
  "mean_pairwise_rmsd_angstrom": 0.9200784389740405,
  "minimum_pairwise_rmsd_angstrom": 0.43051281930916285,
  "maximum_pairwise_rmsd_angstrom": 1.8785258427118026,
  "mean_pairwise_Q": 0.8765852884055132,
  "minimum_pairwise_Q": 0.6914900318829926,
  "maximum_pairwise_Q": 0.9613348489484986,
  "Q_definition": "Nalign^2 / ((1+(RMSD/3A)^2)*N1*N2), using fixed MUSTANG correspondence; not a new PDBeFold alignment",
  "source_coordinates_modified": false,
  "training_membership_modified": false
}


In [2]:
from pathlib import Path
import json,math
import numpy as np
from Bio.PDB import PDBParser
from Bio.SeqUtils import seq1
BASE=Path.cwd();I=BASE/'inputs';O=BASE/'results'
records=json.loads((O/'chimerax_results.json').read_text());cases=json.loads((I/'case_reference.json').read_text());checks=[]
for r in records:
 assert r['status']=='success'
 case=cases[r['Entry']];metadata=json.loads((I/(r['Entry']+'_metadata.json')).read_text())[0]
 assert metadata['sequence']==case['original_sequence']
 residues=[res for res in PDBParser(QUIET=True).get_structure(r['Entry'],I/(r['Entry']+'.pdb')).get_residues() if 'CA' in res]
 assert ''.join(seq1(x.resname) for x in residues)==case['original_sequence']
 assert case['reference_label']==1 and case['score_bits']<17.2 and case['outcome']=='FN'
 ref={x['residue']:np.array(x['coord']) for x in r['reference_trace']};tar={x['residue']:np.array(x['coord']) for x in r['target_trace']}
 squared=[float(np.sum((ref[x['reference_residue']]-tar[x['target_residue']])**2)) for x in r['mapping']];core=[x for x,m in zip(squared,r['mapping']) if m['is_pruned_core']]
 full=math.sqrt(sum(squared)/len(squared));pruned=math.sqrt(sum(core)/len(core));assert abs(full-r['full_rmsd_angstrom'])<1e-4 and abs(pruned-r['core_rmsd_angstrom'])<1e-4
 assert max(core)<=4.0001
 pl=[m['target_pLDDT'] for m in r['mapping'] if m['is_pruned_core']];assert abs(sum(pl)/len(pl)-r['mean_core_pLDDT'])<1e-4
 checks.append(dict(Entry=r['Entry'],split=case['split'],sequence_exactly_matches_project=True,full_pairs=len(squared),pruned_core_pairs=len(core),full_rmsd_independently_verified=full,core_rmsd_independently_verified=pruned,mean_core_pLDDT=r['mean_core_pLDDT']))
assert len(checks)==3
(O/'structure_verification.json').write_text(json.dumps(checks,indent=2));print(json.dumps(checks,indent=2))


[
  {
    "Entry": "D3GGZ8",
    "split": "validation",
    "sequence_exactly_matches_project": true,
    "full_pairs": 53,
    "pruned_core_pairs": 39,
    "full_rmsd_independently_verified": 2.9313678791310673,
    "core_rmsd_independently_verified": 0.8418149476452239,
    "mean_core_pLDDT": 97.02333342723357
  },
  {
    "Entry": "O62247",
    "split": "test",
    "sequence_exactly_matches_project": true,
    "full_pairs": 54,
    "pruned_core_pairs": 39,
    "full_rmsd_independently_verified": 3.3834625228249204,
    "core_rmsd_independently_verified": 0.8167778826119559,
    "mean_core_pLDDT": 95.67410297882863
  },
  {
    "Entry": "A0A1Q1NL17",
    "split": "test",
    "sequence_exactly_matches_project": true,
    "full_pairs": 56,
    "pruned_core_pairs": 43,
    "full_rmsd_independently_verified": 3.4179942131523915,
    "core_rmsd_independently_verified": 0.8731115291390397,
    "mean_core_pLDDT": 91.39999939674554
  }
]
